In [ ]:
from gradio.monitoring_dashboard import total_requests
from rich.jupyter import display
from timm.models.swin_transformer_v2_cr import init_weights
from torch.amp import custom_bwd
from torch.nn.functional import threshold
from transformers.models.x_clip.modeling_x_clip import x_clip_loss
%load_ext autoreload
%autoreload 2

In [ ]:
import logging
from pathlib import Path

import plt
import torch

from core.constants import Constants
from core.settings import DinoDetectorSettings, SamSegmenterSettings, DepthAnythingV2Settings
from image_utils import ImageUtils
from subject_pipeline import SubjectIsolationPipeline
from enhancement_pipeline import EnhancementPipeline

logging.basicConfig(level=logging.DEBUG,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    datefmt='%Y-%m-%d %H:%M')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logging.info(f"DEVICE: {DEVICE}")

dino_settings = DinoDetectorSettings(
    model_config_path="dino/config/GroundingDINO_SwinT_OGC.py",
    model_checkpoint_path="checkpoints/groundingdino_swint_ogc.pth",
    grounding_prompt=Constants.GROUNDING_PROMPT,
    box_threshold=0.30,
    text_threshold=0.30,
)

sam_settings = SamSegmenterSettings(
    checkpoint="checkpoints/sam_vit_b_01ec64.pth",
    sam_model_type="vit_b",
)

depth_settings = DepthAnythingV2Settings(
    encoder="vits",
    features=64,
    out_channels=(48, 96, 192, 384),
    checkpoint_path="checkpoints/depth_anything_v2_vits.pth",
)

seg_subject_pipeline = SubjectIsolationPipeline(
    device=DEVICE,
    dino_settings=dino_settings,
    sam_segmenter_settings=sam_settings,
    depth_anything_settings=depth_settings,
)



In [ ]:
run_folder = True
if run_folder:
    folder_path = Path("photos")
    for file_path in folder_path.iterdir():
        if file_path.is_file():
            subject_region, total_depth, dmin, dmax, image_source = seg_subject_pipeline.find_subject(file_path)
            enhance_pipeline = EnhancementPipeline(
                image=image_source,
                mask=subject_region.mask
            )
            final, clean_mask = enhance_pipeline.run()
            ImageUtils.compare_images(image_source, final)

else:
    file_path = Path("photos/bad_light_1.png")
    subject_region, total_depth, dmin, dmax, image_source = seg_subject_pipeline.find_subject(file_path)
    ImageUtils.show_mask_depth(
        mask=subject_region.mask,
        image_source_depth=total_depth,
        vmin=dmin,
        vmax=dmax,
        title=f"Current region"
    )

    enhance_pipeline = EnhancementPipeline(
        image=image_source,
        mask=subject_region.mask
    )

    final, clean_mask = enhance_pipeline.run()
    ImageUtils.compare_images(image_source, final)

In [ ]:
enhance_pipeline = EnhancementPipeline(
        image=image_source,
        mask=subject_region.mask
    )

final, clean_mask = enhance_pipeline.run()

ImageUtils.compare_images(image_source, final)